# 15_01 — LR range test (발산 임계 진단)

최종 8192 런에 쓸 lr이 **발산 임계 대비 어디에 있는지**만 잰다. 최적 lr을 고르는 도구가 아니다
([ADR-0011](../docs/adr/0011-resource-constrained-methodology.md) 「전이 가능한 lr 판정법」).

신 레시피 `eff_batch 128 · lr 4.8e-4`는 **len512에서만** 안전이 확인됐다(`11_01` 12에폭 완주,
test micro 0.8588). len8192는 같은 eff_batch라도 배치에 담기는 토큰이 16배라 스텝당 기울기
잡음과 발산 임계가 다를 수 있다.

**판정은 절대값이 아니라 상대 위치로 한다.** len512(검증된 조합)를 대조로 함께 재고, len8192에서
운영 lr이 전환점 대비 같은 위치에 있는지 본다 — 경험칙("전환점의 1/3~1/10")을 이 프로젝트의
실측으로 대체한다.

| 프로브 | 조건 | 비용(300 step = 38,400문서 ≈ 0.19 epoch) |
| --- | --- | --- |
| 대조 | len512 · eff128 | 약 7분 |
| 본 | len8192 · eff128 | 약 20~30분 |

⚠️ 프로브는 모델을 발산 구간까지 밀어 넣는다 — **가중치는 버린다.** 여기서 이어서 훈련하지 않는다.

⚠️ **`splits`를 좁히지 않는다.** 훈련 경로는 prep 결과를 `{backbone}_len{max_len}` 캐시에 쓰므로,
train만 prep한 캐시가 남으면 **최종 풀런이 val·test 없이 돈다**. 전체 split을 prep해 두면 8192
prep(비싼 단계)을 최종 런이 그대로 재사용한다.


In [ ]:
import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import (
    TrainingRunner, TrainConfig, probe_batches,
    run_lr_range_test, compare_lr_range,
)

OPERATING_LR = 4.8e-4   # 판정 대상 — 신 레시피의 lr


## 프로브 config

최종 런과 **모든 것을 동일하게** 두고 스케줄만 바꾼다(백본·헤드 초기화·손실·eff_batch·데이터·
샘플러·dtype). lr 축은 `run_lr_range_test`가 지배하므로 `learning_rate`·`warmup_ratio`·`epochs`는
프로브에서 쓰이지 않는 자리값이다.

`micro_batch`만 길이에 따라 다르다 — 8192는 메모리 상한이 낮아 `grad_accum`으로 eff_batch를 맞춘다.
발산 임계는 **eff_batch**에 의존하므로 그 값(128)은 두 프로브에서 같아야 한다.


In [ ]:
def probe_cfg(max_len, micro_batch):
    """LR range test용 config — 최종 런과 같은 레시피 축, 스케줄만 프로브가 대체한다."""
    return TrainConfig(
        backbone="axenc",
        loss="focal",
        loss_params={"alpha": 0.25, "gamma": 2},
        max_len=max_len,
        eff_batch=128,                 # 최종 런과 동일 — 발산 임계는 배치에 의존한다
        micro_batch=micro_batch,       # 메모리 상한(길이에 따라 다름) → grad_accum이 유도됨
        eval_micro_batch=micro_batch,  # 프로브는 eval을 하지 않는다(자리값)
        learning_rate=OPERATING_LR,    # 자리값 — 프로브가 1e-6 → 1e-2로 덮어쓴다
        weight_decay=0.01,             # 최종 런과 동일
        warmup_ratio=0.0,              # 프로브는 warmup 없음
        epochs=1,                      # 자리값 — max_steps가 지배한다
        early_stop_epochs=1,           # 자리값 — early stop 없음
        # splits는 좁히지 않는다 — 위 함정 참조(prep 캐시가 최종 풀런에 재사용된다)
        notebook_name="15_01_LRRangeTest.ipynb",
        tag=f"lrrange-axenc-len{max_len}-b128",
        run_name=f"lrrange_axenc_len{max_len}_eff128",
        repo_final="(unused)",         # push 없음
        out_path=f"/workspace/output/lrrange-len{max_len}",
    )


## 대조 — len512 (검증된 조합)

`11_01`이 이 조합·이 lr로 12에폭을 완주했다. 여기서 읽은 상대 위치가 **"안전하다고 실증된 위치"** 의
기준이 된다.


In [ ]:
cfg512 = probe_cfg(max_len=512, micro_batch=128)
r512 = TrainingRunner(cfg512)
r512.load_data()
r512.prepare_data()
r512.load_model()


In [ ]:
res512 = run_lr_range_test(r512)
res512.summary(operating_lr=OPERATING_LR)
res512.save()


## 본 — len8192

`micro_batch`는 메모리 상한이라 실측으로 정한다. 아래 셀로 안전 상한을 확인한 뒤 `probe_cfg`에 넣는다
(모델만 필요하므로 데이터 prep 없이 돌릴 수 있다).


In [ ]:
# from patent_train import build_model
# from patent_train.backbones import get_backbone
# bb = get_backbone("axenc")
# probe_batches(
#     build_model(bb), 50000, 8192,
#     train_mb=(2, 4, 8, 16),
#     eval_mb=(4, 8, 16),
# )


In [ ]:
cfg8k = probe_cfg(max_len=8192, micro_batch=8)   # OOM 확인 결과로 조정
r8k = TrainingRunner(cfg8k)
r8k.load_data()
r8k.prepare_data()     # 8192 prep — 최종 풀런이 이 캐시를 재사용한다
r8k.load_model()


In [ ]:
res8k = run_lr_range_test(r8k)
res8k.summary(operating_lr=OPERATING_LR)
res8k.save()


## 판정

`compare_lr_range`가 두 조합에서 운영 lr의 상대 위치를 대면시킨다. 판정 규칙은 프로브 전에
못박혀 있다(사후 해석 방지).

- **여유 있음** — 8192의 상대 위치가 512와 비슷하거나 더 낮다 → 신 레시피 그대로 최종 런.
- **경계** — 8192에서 전환점에 뚜렷이 가깝다 → 512와 같은 상대 위치가 되도록 lr을 낮춰 최종 런.
- **위험** — 운영 lr이 전환점 이상 → 8192에서 실증된 유일한 조합인 구 레시피(eff8 · lr 3e-5)로 간다.


In [ ]:
import json, os

verdict = compare_lr_range(control=res512, target=res8k, operating_lr=OPERATING_LR)

os.makedirs("/workspace/output", exist_ok=True)
with open("/workspace/output/lr_range_verdict.json", "w", encoding="utf-8") as f:
    json.dump({
        "note": "LR range test — 신 레시피(eff128·lr 4.8e-4)의 len8192 전이 안전성 진단. "
                "최적 lr 탐색이 아니라 발산 임계 대비 상대 위치 판정이다. "
                "control(len512)은 11_01이 같은 lr로 12에폭 완주한 검증된 조합.",
        "notebook": "notebook/15_01_LRRangeTest.ipynb",
        "control": res512.to_dict(),
        "target": res8k.to_dict(),
        "verdict": verdict,
    }, f, ensure_ascii=False, indent=2)
print("[save] /workspace/output/lr_range_verdict.json")


## 곡선

세 구간을 눈으로 확인한다 — ① 평탄(lr 너무 작음) → ② 하강(학습) → ③ 치솟음(발산).
끝까지 평탄하면 lr 주입이 안 된 것이므로 판정하지 않는다.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for res, style in [(res512, "-"), (res8k, "--")]:
    ax.plot(res.lrs, res.smoothed, style, label=f"len{res.max_len} (smoothed)")
    ax.axvline(res.lr_at_min_loss, ls=":", alpha=0.5)
ax.axvline(OPERATING_LR, color="k", lw=1.5, label=f"operating lr {OPERATING_LR:.1e}")
ax.set_xscale("log"); ax.set_xlabel("learning rate"); ax.set_ylabel("smoothed train loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### verify

프로브가 유효했는지 산출물에서 확인한다.


In [ ]:
# 곡선이 세 구간을 갖는가 — 전환점이 관측 구간의 양 끝이 아니어야 한다
for res in (res512, res8k):
    assert res.lrs[0] < res.lr_at_min_loss < res.lrs[-1],         f"len{res.max_len}: 전환점이 관측 구간 경계에 있다 — start_lr/end_lr 범위를 넓힐 것"
    # 하강이 실제로 일어났는가(평탄 곡선 = lr 주입 실패)
    assert res.smoothed[0] - res.min_smoothed > 0.01 * res.smoothed[0],         f"len{res.max_len}: 손실이 거의 안 내려갔다 — lr 주입·데이터 확인"

# 두 프로브가 같은 레시피 축을 썼는가(eff_batch가 다르면 임계 비교가 성립하지 않는다)
assert res512.eff_batch == res8k.eff_batch == 128

# prep 캐시가 전체 split을 담았는가 — 최종 풀런이 이 캐시를 재사용한다
for r in (r512, r8k):
    assert set(r.data.dataset) == {"train", "val", "test"},         f"len{r.cfg.max_len} prep 캐시에 split이 빠졌다: {sorted(r.data.dataset)}"

print("verify ok — 전환점:",
      f"len512 {res512.lr_at_min_loss:.3e} · len8192 {res8k.lr_at_min_loss:.3e}")
